# 07 · Embedding and searching the chunks

Once the corpus has been divided into chunks, those chunks must be turned into something a
question can search. This notebook describes how the chunks are embedded and indexed, and how a
question then retrieves the most relevant of them.

The relevant code is `ocular/rag/index.py`, `ocular/rag/retrieve.py` and `ocular/rag/rerank.py`.
The indexes were built by `scripts/build_index.py` and, for the full grid, by
`scripts/build_all_indexes.py`. The present notebook loads a finished index and searches it.

## Turning chunks into vectors

Each chunk is passed through an embedding model, which maps it to a fixed-length vector such that
chunks of similar meaning receive nearby vectors. Comparing a question's vector with the chunk
vectors then finds the chunks whose meaning is closest to the question.

The embedding model is not fixed in advance. It is one of the choices the experiment compares, so
three were prepared. A small general baseline (all-MiniLM-L6-v2), a biomedical model fine-tuned on
PubMed literature (MedEmbed-base), and a recent general model from Google (EmbeddingGemma). Because
the chunks can be embedded by any of these, nine indexes exist in total, one for each combination
of the three chunk strategies and the three models. Each index records how it was built in a
manifest.

In [1]:
import json

from ocular.rag import index

json.loads((index.INDEX_DIR / "fixed__MedEmbed-base-v0.1" / "manifest.json").read_text())

{'model_name': 'abhinand/MedEmbed-base-v0.1',
 'dim': 768,
 'n_chunks': 15203,
 'created_at': '2026-08-25T13:01:34.169294+00:00',
 'chunk_strategy': 'fixed'}

## Two representations of the same chunks

An index holds two representations of its chunks, because they retrieve in complementary ways. The
**dense** representation is the set of embedding vectors described above, which capture meaning and
can connect a question to a chunk even when the two share no words. The **sparse** representation is
a BM25 lexical index, which rewards exact overlap of terms and so captures precise wording and
abbreviations, such as CNV or OCT, that embeddings tend to blur.

Only the embeddings are stored on disk. The BM25 index is inexpensive and deterministic, so it is
rebuilt from the chunk texts each time an index is loaded.

## Three ways to retrieve

Because there are two representations, there are three ways to retrieve. The **vector** mode ranks
chunks by the cosine similarity of their embeddings to the question. The **bm25** mode ranks them
by their lexical score. The **hybrid** mode runs both and combines the two rankings with Reciprocal
Rank Fusion, which merges two ranked lists by adding, for each chunk, a score that decreases with
its position in each list. Working on positions rather than on the raw scores means the fusion
needs no calibration between the bounded cosine scale and the unbounded BM25 scale.

In [2]:
from ocular.rag import retrieve

idx = index.load_index(index.INDEX_DIR / "fixed__MedEmbed-base-v0.1")
query = "What is the first-line treatment for diabetic macular edema?"

for mode in ("vector", "bm25", "hybrid"):
    print(f"[{mode}]")
    for h in retrieve.retrieve(idx, query, k=3, mode=mode):
        print(f"   {h.chunk.title[:60]}")
    print()

[vector]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Comparison of adverse renal events between ranibizumab and a
   Telangiectatic capillaries remodel over time in eyes with pe
   Effect of calcium dobesilate on intraretinal cysts and exuda

[bm25]
   Direct randomized evidence comparing ranibizumab and bevaciz
   Therapeutic potential of prolactin-releasing anti-dopaminerg
   Effects of anti-VEGF therapy on choroidal thickness in eyes 

[hybrid]
   Telangiectatic capillaries remodel over time in eyes with pe
   Comparison of adverse renal events between ranibizumab and a
   Effects of anti-VEGF therapy on choroidal thickness in eyes 



## Reranking the candidates

First-stage retrieval is fast because it compares vectors that were computed independently, but
that independence limits its precision. A reranker instead reads the question and a candidate
together and scores their relevance directly, which is far more accurate but too slow to apply to
the whole corpus. The standard remedy, adopted here, is to retrieve a wide set of candidates and
then rerank only those, keeping the best few.

The reranker is a choice separate from the embedding model, and the experiment found it to be the
single most influential component of retrieval. A small ModernBERT reranker,
gte-reranker-modernbert-base, was selected. The cell below shows how reranking reorders the
candidates for the same question.

In [3]:
from ocular.rag import rerank

candidates = retrieve.retrieve(idx, query, k=20, mode="hybrid")
print("top 3 before reranking")
for h in candidates[:3]:
    print(f"   {h.chunk.title[:60]}")

reranked = rerank.rerank(query, candidates, k=3)
print("\ntop 3 after reranking")
for h in reranked:
    print(f"   {h.chunk.title[:60]}")

top 3 before reranking
   Telangiectatic capillaries remodel over time in eyes with pe
   Comparison of adverse renal events between ranibizumab and a
   Effects of anti-VEGF therapy on choroidal thickness in eyes 


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]


top 3 after reranking
   Comparison of adverse renal events between ranibizumab and a
   Effects of anti-VEGF therapy on choroidal thickness in eyes 
   Direct randomized evidence comparing ranibizumab and bevaciz


## How the indexing and retrieval are implemented

The pieces demonstrated above are implemented in `ocular/rag/index.py`, `ocular/rag/retrieve.py` and
`ocular/rag/rerank.py`. This section describes what each one does behind the scenes rather than
reproducing the code.

**Building an index.** The function `build_index` loads the chosen embedding model through
`load_embedder`, embeds every chunk text into one matrix, and writes three files to the index
directory. These are `embeddings.npy`, the dense matrix of chunk vectors, `chunks.jsonl`, the chunks
themselves aligned row for row with that matrix, and `manifest.json`, which records the model, the
vector dimension, the chunk count, a timestamp and the chunking configuration. The lexical side is
deliberately not saved.

**Loading an index.** The function `load_index` reads those three files back into an `Index` object
and rebuilds the BM25 lexical side from the chunk texts with `_build_bm25`, since that side is cheap
to construct and deterministic. The `Index` therefore carries both representations together, the dense
embeddings and the BM25 model, along with the name of the embedding model so that a query can be
encoded the same way it was at build time.

**Retrieving.** The entry point `retrieve` dispatches on the mode. In `vector_search` the query is
embedded by the index's own model and scored against the chunk vectors by cosine similarity. In
`bm25_search` the query is scored lexically. In `hybrid_search` each retriever contributes its top
candidates, and the two rankings are combined by Reciprocal Rank Fusion, in which a chunk receives
`1 / (rrf_k + rank)` from each list where it appears and the fused scores are then sorted. Because the
fusion adds ranks rather than raw scores, it needs no calibration between the bounded cosine scale and
the unbounded BM25 scale.

**Reranking.** The function `rerank` loads the cross-encoder through `load_reranker`, scores each
question and candidate pair jointly, and returns the best few. It runs only on the wide candidate set
produced by first-stage retrieval, which is what keeps an accurate but slow model affordable.

## What was run

Each index was built by embedding a strategy's chunks with one model and writing the result
together with its manifest. The command for a single index is

    python scripts/build_index.py --chunk-strategy fixed --embed-model abhinand/MedEmbed-base-v0.1 --device cpu

and all nine indexes were built by the driver `scripts/build_all_indexes.py`, which walks every
strategy and model and skips whatever already exists. The `--device cpu` argument is present
because, on the machine used, the larger embedding models stall on the Apple GPU backend during the
full-corpus embedding, so they are pinned to the processor, which is slower but reliable.

The command shown above is the one that produced the very index inspected near the top of this
notebook. It wrote the `manifest.json` reported there, recording the `abhinand/MedEmbed-base-v0.1`
model, a vector dimension of 768 and the 15,203 fixed chunks, so the artifact fully describes the
configuration that made it.

## Summary

- Each chunk is embedded into a vector, and every index also carries a BM25 lexical view of the
  same chunks.
- A question can retrieve by **vector**, by **bm25**, or by their **hybrid** fusion.
- A **reranker** then rescores the strongest candidates, and it is the most influential single
  component of retrieval.

The next notebook, `08`, evaluates these choices and reports which configuration was selected and
why.